# Lab 3: Erste Crew mit CrewAI (Analyst, Redakteur, Flow)

**Lernziel.** Sie bauen eine erste Crew aus zwei Agenten, die aus einem Git-Diff eine Änderungszusammenfassung für ein Changelog erzeugt. Sie erzwingen mit `output_pydantic` eine strukturierte Ausgabe, sichern sie mit einem Guardrail ab und vergleichen den sequentiellen mit dem hierarchischen Prozess. Danach betten Sie dieselbe Crew in einen Flow ein, der über State und Router entscheidet, ob die Crew überhaupt läuft, und der die Freigabe an einen Menschen zurückgibt.

Eingabe ist ein Diff als Text aus `data/diff_rabatt.patch` (Rabattstaffel mit einem Fehler an einer Grenze, ohne Test) bzw. `data/diff_leer.patch` (leere Datei). Die Diffs werden bewusst aus Dateien gelesen, damit das Lab ohne Git-Repo und ohne Netz läuft.

Erwartete Ergebnisse stehen in `EXPECTED_RESULTS.md`.

In [ ]:
import os, time, json
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

ENV_FILE = find_dotenv(usecwd=True)          # sucht .env ab dem Notebook-Ordner (labs/.env)
print("Konfiguration aus:", ENV_FILE or "keine .env gefunden, Defaults (LM Studio)")
load_dotenv(ENV_FILE, override=True)   # override: CrewAI lädt beim Import schon eine .env aus einem Elternordner
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:1234/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "lm-studio")
LLM_MODEL = os.environ.get("LLM_MODEL", "qwen/qwen3.6-35b-a3b")
# Cloud-Schalter (nur als Beispiel, nicht aufrufen):
# LLM_BASE_URL=https://api.openai.com/v1  LLM_API_KEY=sk-...  LLM_MODEL=gpt-4.1-mini

from pydantic import BaseModel
from crewai import Agent, Task, Crew, Process, LLM, TaskOutput

DATA = Path("data")
diff_text = (DATA / "diff_rabatt.patch").read_text(encoding="utf-8")
print(f"Diff geladen: {len(diff_text.splitlines())} Zeilen")
print(diff_text[:600])

# Verbindungstest: ein kurzer Aufruf gegen den Endpunkt
llm = LLM(model=f"openai/{LLM_MODEL}", base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)
print("Modell:", llm.model, "| Endpunkt:", LLM_BASE_URL)
print("LLM antwortet:", llm.call("Antworte nur mit OK.")[:40])

In [ ]:
# Strukturierte Ausgabe (output_pydantic) schickt CrewAI als response_format=json_schema an den Endpunkt.
# LM Studio und OpenAI können das; manche OpenAI-kompatible Endpunkte (z. B. DeepSeek) antworten mit 400.
# Dann fällt dieser Wrapper auf Text zurück und parst das JSON selbst in das Pydantic-Modell.
import re
from openai import BadRequestError
from crewai.llms.providers.openai.completion import OpenAICompletion


class JsonFallbackLLM(OpenAICompletion):
    native_json: bool = True   # wird nach dem ersten 400 auf False gesetzt

    def _fallback(self, text, response_model):
        m = re.search(r"\{.*\}", str(text), re.S)
        return response_model.model_validate_json(m.group(0)) if m else text

    def call(self, messages, tools=None, callbacks=None, available_functions=None,
             from_task=None, from_agent=None, response_model=None):
        if response_model is not None and self.native_json:
            try:
                return super().call(messages, tools, callbacks, available_functions, from_task, from_agent, response_model)
            except BadRequestError as e:
                if "response_format" not in str(e):
                    raise
                self.native_json = False
        text = super().call(messages, tools, callbacks, available_functions, from_task, from_agent, None)
        return self._fallback(text, response_model) if response_model is not None else text

    async def acall(self, messages, tools=None, callbacks=None, available_functions=None,
                    from_task=None, from_agent=None, response_model=None):
        if response_model is not None and self.native_json:
            try:
                return await super().acall(messages, tools, callbacks, available_functions, from_task, from_agent, response_model)
            except BadRequestError as e:
                if "response_format" not in str(e):
                    raise
                self.native_json = False
        text = await super().acall(messages, tools, callbacks, available_functions, from_task, from_agent, None)
        return self._fallback(text, response_model) if response_model is not None else text


class _Probe(BaseModel):
    ok: bool


import logging
logging.disable(logging.ERROR)          # der Probe-Aufruf darf still scheitern
try:
    llm.call([{"role": "user", "content": 'Antworte mit {"ok": true}'}], response_model=_Probe)
    print("Endpunkt unterstützt json_schema: LLM bleibt wie definiert")
except BadRequestError as e:
    if "response_format" not in str(e):
        raise
    llm = JsonFallbackLLM(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)
    llm.native_json = False
    print("Endpunkt ohne json_schema-Unterstützung: JsonFallbackLLM aktiv (JSON wird aus Text geparst)")
finally:
    logging.disable(logging.NOTSET)


def make_llm():
    """Frische LLM-Instanz. CrewAI zählt token_usage pro LLM-Instanz über deren Lebensdauer;
    je Agent eine eigene Instanz hält die Zahlen je Crew-Lauf sauber."""
    if isinstance(llm, JsonFallbackLLM):
        fresh = JsonFallbackLLM(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)
        fresh.native_json = False
        return fresh
    return LLM(model=f"openai/{LLM_MODEL}", base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)

## Aufgabe 1: LLM und zwei Agenten definieren

**Was.** Definieren Sie zwei Agenten: einen **Code Analyst**, der den Diff liest und Änderungen sowie Risiken benennt, und einen **Technical Writer**, der daraus eine Änderungszusammenfassung für ein Changelog schreibt. Jeder Agent bekommt `role`, `goal`, `backstory`, das `llm` aus der Setup-Zelle, `max_iter=3` und `verbose=True`.

**Warum.** `role`, `goal` und `backstory` landen wörtlich im System-Prompt. Kurz und konkret schlägt lang und blumig: Das Modell soll wissen, worauf es achten soll (Grenzfälle, fehlende Tests), nicht, wie toll es ist. `max_iter` begrenzt die Schleife, `verbose=True` zeigt beim ersten Lauf, was der Agent tatsächlich vom Modell bekommt.

Packen Sie die Definition in eine Funktion `make_agents()`, die `(analyst, writer)` zurückgibt, und geben Sie jedem Agenten eine eigene LLM-Instanz aus `make_llm()` (Setup-Zelle). Grund: CrewAI zählt Tokens pro LLM-Instanz über deren gesamte Lebensdauer und summiert für `result.token_usage` die Instanzen aller Agenten der Crew. Teilen sich zwei Agenten eine Instanz, zählt die Crew sie doppelt; wird die Instanz in mehreren Crews benutzt, misst die zweite Crew die Summe. Frische Agenten mit eigener Instanz je Crew-Lauf halten die Zahlen vergleichbar (Aufgabe 4) und machen den Flow (Aufgabe 5) unabhängig von vorherigen Zellen.

**Woran erkennen Sie Erfolg.** `analyst.role` und `writer.role` sind gesetzt, `analyst.llm.model` zeigt Ihr Modell. Der eigentliche Test kommt in Aufgabe 2.

Hinweis: Rollen und Ziele sind auf Englisch formuliert, weil kleine lokale Modelle damit stabiler arbeiten. Die Ausgabe (Changelog) verlangen wir in Aufgabe 2 trotzdem auf Deutsch.

In [ ]:
def make_agents():
    """Liefert (analyst, writer) als frische Agent-Objekte."""
    # TODO: Agent "Code Analyst": role, goal (Diff lesen, Änderungen und Risiken benennen, Grenzfälle prüfen),
    #       backstory (1 bis 2 Sätze), llm=make_llm(), max_iter=3, verbose=True
    analyst = None
    # TODO: Agent "Technical Writer": role, goal (Changelog-Eintrag aus der Analyse), backstory kurz,
    #       llm=make_llm(), max_iter=3, verbose=True
    writer = None
    return analyst, writer


# analyst, writer = make_agents()
# print(analyst.role, "|", writer.role, "|", analyst.llm.model)

### Lösung

In [ ]:
# LÖSUNG
# Englische Rollen/Ziele: kleine lokale Modelle folgen englischen System-Prompts stabiler.
def make_agents():
    """Liefert (analyst, writer) als frische Agenten mit je eigener LLM-Instanz."""
    analyst = Agent(
        role="Code Analyst",
        goal=(
            "Read a unified diff and list every functional change and every risk: "
            "bugs, boundary conditions, missing tests. Point to function names."
        ),
        backstory=(
            "Senior developer doing code review. Precise and skeptical; "
            "checks each comparison operator against the documented rule."
        ),
        llm=make_llm(),
        max_iter=3,
        verbose=True,
    )
    writer = Agent(
        role="Technical Writer",
        goal="Turn a technical analysis into a short changelog entry that a developer can act on.",
        backstory="Writes release notes. One short sentence per item, no marketing language, no speculation.",
        llm=make_llm(),
        max_iter=3,
        verbose=True,
    )
    return analyst, writer


analyst, writer = make_agents()
print(analyst.role, "|", writer.role, "|", analyst.llm.model)

## Aufgabe 2: Tasks mit `context` und strukturierte Ausgabe

**Was.** Zwei Tasks: Task 1 (Analyse) mit dem Diff als Platzhalter `{diff}` in der `description` und `expected_output` als zwei Listen (Änderungen, Risiken). Task 2 (Zusammenfassung) mit `context=[task1]`, damit die Analyse als Eingabe ankommt, und `output_pydantic=ChangeSummary`. Definieren Sie `ChangeSummary` als Pydantic-Modell mit `title: str`, `changes: list[str]`, `risks: list[str]`, `needs_tests: bool`. Bauen Sie die Tasks in einer Funktion `make_tasks()`, damit spätere Aufgaben frische Task-Objekte bekommen (ein Task speichert seine Ausgabe, deshalb nicht wiederverwenden). Dann `Crew(process=Process.sequential)` starten, `result.pydantic` inspizieren und `result.token_usage` ausgeben.

**Notebook-Besonderheit.** In einem Skript heißt der Aufruf `crew.kickoff(inputs=...)`. Im Notebook läuft bereits ein Event-Loop (Jupyter), und CrewAI 1.15 weigert sich dann, synchron zu starten (`RuntimeError: Agent execution was invoked synchronously from within a running event loop`). Deshalb hier `result = await crew.kickoff_async(inputs=...)`; Jupyter erlaubt `await` direkt in der Zelle. Innerhalb eines Flows (Aufgabe 5) geht wieder das normale `kickoff()`, weil der Flow seine Methoden in einem eigenen Thread ausführt.

**Warum.** `context` ist die explizite Datenleitung zwischen Tasks; ohne sie sieht der Writer die Analyse nicht. `output_pydantic` macht aus Prosa ein Objekt, das der Rest Ihres Programms typsicher verarbeiten kann. `token_usage` ist der Kostenzähler, den Sie im Betrieb brauchen.

**Woran erkennen Sie Erfolg.** `result.pydantic` ist ein `ChangeSummary`; `changes` und `risks` sind nicht leer; in `risks` taucht die Grenze bei 100 bzw. 500 EUR auf (`>` statt `>=` in `rabattsatz`) und der fehlende Test. `result.token_usage.total_tokens` ist eine Zahl größer null.

In [ ]:
class ChangeSummary(BaseModel):
    # TODO: title: str, changes: list[str], risks: list[str], needs_tests: bool
    pass


def make_tasks(analyst, writer, **task2_kwargs):
    """Liefert (task1, task2). task2_kwargs erlaubt später z. B. guardrail=..."""
    task1 = Task(
        # TODO: description mit {diff}-Platzhalter, expected_output (zwei Listen), agent=analyst
    )
    task2 = Task(
        # TODO: description (Changelog auf Deutsch), expected_output, agent=writer,
        #       context=[task1], output_pydantic=ChangeSummary, **task2_kwargs
    )
    return task1, task2


# TODO: Crew mit beiden Agenten, beiden Tasks, Process.sequential, verbose=True
# TODO: result = await crew.kickoff_async(inputs={"diff": diff_text}); result.pydantic und result.token_usage ausgeben

### Lösung

In [ ]:
# LÖSUNG
class ChangeSummary(BaseModel):
    title: str
    changes: list[str]
    risks: list[str]
    needs_tests: bool


def make_tasks(analyst, writer, **task2_kwargs):
    """Liefert (task1, task2). task2_kwargs erlaubt später z. B. guardrail=..."""
    task1 = Task(
        description=(
            "Analyse this unified diff:\n\n{diff}\n\n"
            "List every functional change and every risk. Check boundary conditions "
            "against the comments in the code and check whether tests were added."
        ),
        expected_output=(
            "Two bullet lists titled 'Changes' and 'Risks'. "
            "Each bullet is one sentence of at most 20 words and names the affected function."
        ),
        agent=analyst,
    )
    task2 = Task(
        description=(
            "Write a changelog entry from the analysis you received as context. "
            "Write title, changes and risks in German. Each item in changes and risks is "
            "one sentence of at most 120 characters. "
            "Set needs_tests to true if the analysis reports missing tests."
        ),
        expected_output=(
            "A ChangeSummary: title (one line), changes (list of short sentences), "
            "risks (list of short sentences), needs_tests (bool)."
        ),
        agent=writer,
        context=[task1],
        output_pydantic=ChangeSummary,
        **task2_kwargs,
    )
    return task1, task2


task1, task2 = make_tasks(analyst, writer)
crew = Crew(agents=[analyst, writer], tasks=[task1, task2], process=Process.sequential, verbose=True)

t0 = time.time()
result = await crew.kickoff_async(inputs={"diff": diff_text})   # im Skript: crew.kickoff(...)
seq_seconds = time.time() - t0

summary = result.pydantic
print(type(summary).__name__)
print(json.dumps(summary.model_dump(), indent=2, ensure_ascii=False))
print(f"\nLaufzeit sequential: {seq_seconds:.0f} s")
print("Tokens:", result.token_usage)
seq_tokens = result.token_usage.total_tokens
analysis_text = result.tasks_output[0].raw   # Analyse-Text für Aufgabe 3 aufheben

## Aufgabe 3: Guardrail an Task 2

**Was.** Schreiben Sie eine Guardrail-Funktion `check_summary(output: TaskOutput) -> tuple[bool, Any]`, die prüft: `changes` ist nicht leer, kein Eintrag in `changes` oder `risks` ist länger als `MAX_LEN` Zeichen, der Titel enthält keinen Zeilenumbruch. Bei Verstoß `(False, "Begründung")`, sonst `(True, ...)`. Hängen Sie sie mit `guardrail=check_summary, guardrail_max_retries=2` an Task 2. Provozieren Sie zuerst einen Verstoß mit `MAX_LEN = 20`, lesen Sie das Retry-Verhalten im Log, dann setzen Sie eine sinnvolle Grenze (`MAX_LEN = 200`).

Beachten Sie: CrewAI hängt bei `output_pydantic` die Anweisung an, den Inhalt „exactly as-is“ zu übernehmen. Die Länge der Einträge wird also schon vom Analysten bestimmt; deshalb begrenzt `make_tasks()` bereits dessen Bullets auf 20 Wörter und den Writer auf 120 Zeichen je Eintrag.

Damit die Zelle nicht jedes Mal den Analysten neu laufen lässt, bekommt der Writer hier die gespeicherte Analyse aus Aufgabe 2 (`analysis_text`) direkt in die `description` (Platzhalter `{analysis}`) und läuft allein in einer Crew.

**Warum.** Ein Guardrail ist deterministischer Code zwischen Modell und Weiterverarbeitung. Bei `(False, ...)` schickt CrewAI die Begründung als Feedback an den Agenten und lässt ihn neu antworten, bis `guardrail_max_retries` erreicht ist; danach wirft `kickoff()` eine Exception. Genau dieses Verhalten sehen Sie im Log als `Guardrail blocked (attempt 1/3), retrying due to: ...`.

Zwei Details, die man wissen muss: Der Guardrail bekommt ein `TaskOutput`; ob `output.pydantic` schon gefüllt ist, hängt davon ab, ob das Modell direkt ein strukturiertes Objekt geliefert hat. Deshalb parsen Sie zur Sicherheit `output.raw` selbst, wenn `output.pydantic` `None` ist. Und: Bei Erfolg geben Sie den geprüften Inhalt als JSON-String zurück; CrewAI baut daraus wieder das Pydantic-Objekt.

**Woran erkennen Sie Erfolg.** Mit `MAX_LEN = 20` mindestens eine gelbe `Guardrail blocked`-Zeile im Log (Lauf endet mit Exception oder mit stark verkürzten Einträgen). Mit `MAX_LEN = 200` läuft die Crew ohne Retry durch und `result.pydantic` ist ein gültiges `ChangeSummary`.

In [ ]:
MAX_LEN = 20   # absichtlich zu streng, später 200


def check_summary(output: TaskOutput):
    """Guardrail: (True, json_string) oder (False, 'Begründung')."""
    # TODO: ChangeSummary aus output.pydantic oder ChangeSummary.model_validate_json(output.raw) holen
    #       (bei Parse-Fehler (False, "kein gültiges ChangeSummary-JSON: ..."))
    # TODO: changes leer -> (False, ...)
    # TODO: Eintrag in changes + risks länger als MAX_LEN -> (False, ...)
    # TODO: "\n" im Titel -> (False, ...)
    # TODO: return (True, cs.model_dump_json())
    pass


async def run_writer_with_guardrail(analysis: str):
    """Nur der Writer läuft; die Analyse kommt als Text in die description."""
    task = Task(
        # TODO: description mit {analysis}-Platzhalter, expected_output, agent=writer,
        #       output_pydantic=ChangeSummary, guardrail=check_summary, guardrail_max_retries=2
    )
    # TODO: await Crew(agents=[writer], tasks=[task], verbose=True).kickoff_async(inputs={"analysis": analysis})
    #       Exception abfangen und ausgeben (nach 2 Retries wirft kickoff)


# await run_writer_with_guardrail(analysis_text)

### Lösung

In [ ]:
# LÖSUNG
MAX_LEN = 20   # absichtlich zu streng, unten auf 200 gesetzt


def check_summary(output: TaskOutput):
    """Guardrail: (True, json_string) oder (False, 'Begründung')."""
    try:
        cs = output.pydantic or ChangeSummary.model_validate_json(output.raw)
    except Exception as e:
        return (False, f"Kein gültiges ChangeSummary-JSON: {e}")
    if not cs.changes:
        return (False, "changes darf nicht leer sein")
    for entry in cs.changes + cs.risks:
        if len(entry) > MAX_LEN:
            return (False, f"Eintrag mit {len(entry)} Zeichen überschreitet {MAX_LEN}: '{entry[:40]}...'. "
                           f"Kürze JEDEN Eintrag in changes und risks auf höchstens {MAX_LEN} Zeichen.")
    if "\n" in cs.title:
        return (False, "Titel darf keinen Zeilenumbruch enthalten")
    return (True, cs.model_dump_json())   # String -> CrewAI baut daraus wieder das Pydantic-Objekt


async def run_writer_with_guardrail(analysis: str):
    """Nur der Writer läuft; die Analyse kommt als Text in die description."""
    task = Task(
        description=(
            "Write a changelog entry from this analysis:\n\n{analysis}\n\n"
            "Write title, changes and risks in German. Each item in changes and risks is "
            "one sentence of at most 120 characters. "
            "Set needs_tests to true if the analysis reports missing tests."
        ),
        expected_output="A ChangeSummary: title, changes (list), risks (list), needs_tests (bool).",
        agent=writer,
        output_pydantic=ChangeSummary,
        guardrail=check_summary,
        guardrail_max_retries=2,
    )
    crew = Crew(agents=[writer], tasks=[task], process=Process.sequential, verbose=True)
    t0 = time.time()
    try:
        res = await crew.kickoff_async(inputs={"analysis": analysis})
        print(f"\nDurchgelaufen in {time.time() - t0:.0f} s, Retries: {task.retry_count}")
        print(json.dumps(res.pydantic.model_dump(), indent=2, ensure_ascii=False))
        return res
    except Exception as e:
        print(f"\nAbbruch nach {time.time() - t0:.0f} s, Retries: {task.retry_count}")
        print("Exception:", str(e)[:300])
        return None


print("=== Lauf 1: MAX_LEN = 20 (Verstoß erwartet) ===")
await run_writer_with_guardrail(analysis_text)

In [ ]:
# LÖSUNG
MAX_LEN = 200   # sinnvolle Grenze: ein Satz, der auf eine Changelog-Zeile passt
print("=== Lauf 2: MAX_LEN = 200 ===")
guarded = await run_writer_with_guardrail(analysis_text)
assert guarded is not None and guarded.pydantic.changes, "Guardrail-Lauf sollte mit MAX_LEN=200 durchlaufen"

## Aufgabe 4 (optional): Hierarchischer Prozess

**Was.** Frische Agenten aus `make_agents()` (sonst zählt `token_usage` die vorigen Läufe mit) und frische Tasks aus `make_tasks()`, aber `process=Process.hierarchical` und `manager_llm=make_llm()`. CrewAI erzeugt dann einen Manager-Agenten, der die Tasks an Analyst und Writer delegiert (Werkzeug `Delegate work to coworker`) und deren Ergebnisse prüft. Messen Sie Laufzeit und `token_usage` und vergleichen Sie mit dem sequentiellen Lauf aus Aufgabe 2.

**Warum.** Der hierarchische Prozess kauft Flexibilität (der Manager entscheidet, wer was macht) mit zusätzlichen Modellaufrufen: jede Delegation ist ein Tool-Call des Managers plus ein kompletter Agentenlauf. Für eine feste Pipeline wie diese ist das reine Mehrkosten; deshalb ist in Lab 4 ein Flow mit sequentiellen Crews der Rahmen, nicht der Manager.

**Woran erkennen Sie Erfolg.** Im Log tauchen `Crew Manager` und Delegations-Tool-Calls auf (oft scheitert der erste Delegationsversuch an der Argumentprüfung des Tools, der Manager korrigiert sich). `hier_tokens` liegt deutlich über `seq_tokens` (Faktor 3 und mehr), die Laufzeit ebenfalls. Wenn das lokale Modell mit dem Manager-Prompt nicht zurechtkommt (Endlosdelegation, `max_iter` erreicht, kein Pydantic-Ergebnis), ist das ein Ergebnis für sich: notieren Sie es und gehen Sie weiter. Die Zelle ist deshalb als optional gekennzeichnet.

In [ ]:
# TODO: analyst_h, writer_h = make_agents(); task1h, task2h = make_tasks(analyst_h, writer_h)
# TODO: await Crew(..., process=Process.hierarchical, manager_llm=make_llm(), verbose=True).kickoff_async(inputs={"diff": diff_text})
# TODO: Laufzeit und token_usage.total_tokens gegen seq_seconds / seq_tokens ausgeben

### Lösung

In [ ]:
# LÖSUNG (optional: mit kleinen lokalen Modellen kann der Manager scheitern, dann Ausgabe lesen und weitergehen)
analyst_h, writer_h = make_agents()          # frische Objekte: token_usage startet bei null
task1h, task2h = make_tasks(analyst_h, writer_h)
hier_crew = Crew(
    agents=[analyst_h, writer_h],
    tasks=[task1h, task2h],
    process=Process.hierarchical,
    manager_llm=make_llm(),                  # eigene Instanz, damit die Manager-Tokens sauber dazukommen
    verbose=True,
)
t0 = time.time()
try:
    hier_result = await hier_crew.kickoff_async(inputs={"diff": diff_text})
    hier_seconds = time.time() - t0
    hier_tokens = hier_result.token_usage.total_tokens
    print("\nHierarchisch, Ergebnis:", hier_result.pydantic or hier_result.raw[:300])
    print(f"sequential:   {seq_seconds:6.0f} s, {seq_tokens:7d} Tokens")
    print(f"hierarchical: {hier_seconds:6.0f} s, {hier_tokens:7d} Tokens  (Faktor {hier_tokens / seq_tokens:.1f})")
except Exception as e:
    print(f"\nHierarchischer Lauf abgebrochen nach {time.time() - t0:.0f} s: {str(e)[:300]}")

## Aufgabe 5: Dieselbe Crew in einem Flow mit State und Routing

**Was.** Ein `ChangelogState(BaseModel)` mit `diff_path: str = ""`, `diff: str = ""`, `summary: ChangeSummary | None = None`. Ein `ChangelogFlow(Flow[ChangelogState])` mit vier Methoden:

- `@start() load_diff`: liest die Datei aus `self.state.diff_path` in `self.state.diff`.
- `@router(load_diff) route`: gibt `"empty"` zurück, wenn der Diff leer ist, sonst `"has_changes"`.
- `@listen("has_changes") run_crew`: baut die Crew (frische Tasks über `make_tasks()`), ruft `kickoff(inputs={"diff": self.state.diff})` auf, legt `result.pydantic` in `self.state.summary` und gibt es zurück.
- `@listen("empty") skip`: gibt einen Hinweis-String zurück, ohne ein Modell aufzurufen.

Starten Sie den Flow mit `flow.kickoff(inputs={"diff_path": ...})` einmal für `diff_rabatt.patch` und einmal für `diff_leer.patch`. Rufen Sie `flow.plot("changelog_flow.html", show=False)` auf. CrewAI 1.15 schreibt HTML, CSS und JS in ein temporäres Verzeichnis und gibt den Pfad zurück; die Hilfsfunktion `save_plot()` in der Lösung fügt CSS und JS in die HTML-Datei ein und legt `changelog_flow.html` als eine Datei in `labs/` ab (Nebenprodukt, kann gelöscht werden; braucht zum Anzeigen Netz für Schriften und Icons).

**Warum.** Der Flow ist der deterministische Rahmen: Dateizugriff, Leerprüfung und Routing sind normaler Python-Code, kein Modellaufruf. Nur der Schritt, der wirklich Sprachverständnis braucht, ruft die Crew. Der State ist typisiert, deshalb sehen Sie nach dem Lauf sofort, was gefüllt wurde. Genau so wird in Lab 4 die Review-Pipeline aufgebaut.

`flow.kickoff()` funktioniert im Notebook direkt: Der Flow erkennt den laufenden Event-Loop und führt sich in einem eigenen Thread aus; deshalb darf `run_crew` darin auch das normale `crew.kickoff()` verwenden.

**Woran erkennen Sie Erfolg.** Lauf 1: Router gibt `has_changes`, `flow.state.summary` ist ein `ChangeSummary`. Lauf 2: Router gibt `empty`, Rückgabe ist der Hinweis, `summary` bleibt `None`, kein Agent wurde gestartet (kein `Agent Started` im Log). Die Datei `changelog_flow.html` existiert.

In [ ]:
from crewai.flow.flow import Flow, start, listen, router


class ChangelogState(BaseModel):
    # TODO: diff_path: str = "", diff: str = "", summary: ChangeSummary | None = None
    pass


class ChangelogFlow(Flow[ChangelogState]):
    # TODO: @start() load_diff  -> Datei aus self.state.diff_path lesen, in self.state.diff ablegen
    # TODO: @router(load_diff) route -> "empty" oder "has_changes"
    # TODO: @listen("has_changes") run_crew -> Crew bauen, kickoff, result.pydantic in self.state.summary
    # TODO: @listen("empty") skip -> Hinweis zurückgeben
    pass


# TODO: flow = ChangelogFlow(); flow.kickoff(inputs={"diff_path": str(DATA / "diff_rabatt.patch")}); flow.state ausgeben
# TODO: zweiter Lauf mit diff_leer.patch
# TODO: flow.plot("changelog_flow.html", show=False) und Ergebnis mit save_plot() nach labs/ kopieren

### Lösung

In [ ]:
# LÖSUNG
from crewai.flow.flow import Flow, start, listen, router


class ChangelogState(BaseModel):
    diff_path: str = ""
    diff: str = ""
    summary: ChangeSummary | None = None


def build_crew(verbose: bool = False) -> Crew:
    a, w = make_agents()                      # frische Agenten je Lauf
    t1, t2 = make_tasks(a, w)
    return Crew(agents=[a, w], tasks=[t1, t2], process=Process.sequential, verbose=verbose)


class ChangelogFlow(Flow[ChangelogState]):

    @start()
    def load_diff(self):
        self.state.diff = Path(self.state.diff_path).read_text(encoding="utf-8")
        print(f"[load_diff] {self.state.diff_path}: {len(self.state.diff)} Zeichen")

    @router(load_diff)
    def route(self):
        decision = "empty" if not self.state.diff.strip() else "has_changes"
        print(f"[route] -> {decision}")
        return decision

    @listen("has_changes")
    def run_crew(self):
        result = build_crew().kickoff(inputs={"diff": self.state.diff})
        self.state.summary = result.pydantic
        print(f"[run_crew] {result.token_usage.total_tokens} Tokens")
        return self.state.summary

    @listen("empty")
    def skip(self):
        print("[skip] Diff ist leer, keine Crew gestartet")
        return "Kein Changelog-Eintrag: Diff ist leer."


print("=== Lauf 1: diff_rabatt.patch ===")
flow = ChangelogFlow()
t0 = time.time()
out = flow.kickoff(inputs={"diff_path": str(DATA / "diff_rabatt.patch")})
print(f"Rückgabe ({type(out).__name__}) nach {time.time() - t0:.0f} s:")
print(json.dumps(flow.state.summary.model_dump(), indent=2, ensure_ascii=False))

print("\n=== Lauf 2: diff_leer.patch ===")
flow2 = ChangelogFlow()
out2 = flow2.kickoff(inputs={"diff_path": str(DATA / "diff_leer.patch")})
print("Rückgabe:", out2)
print("summary im State:", flow2.state.summary)

import shutil


def save_plot(flow: Flow, name: str) -> Path:
    """plot() schreibt html+css+js in ein Temp-Verzeichnis; hier als eine Datei nach ./name.html kopieren."""
    tmp = Path(flow.plot(f"{name}.html", show=False))
    if tmp.suffix != ".html":            # plot() gibt den Pfad ohne Endung zurück
        tmp = tmp.with_name(tmp.name + ".html") if tmp.with_name(tmp.name + ".html").exists() else tmp
    html = tmp.read_text(encoding="utf-8")
    css = (tmp.parent / f"{name}_style.css").read_text(encoding="utf-8")
    js = (tmp.parent / f"{name}_script.js").read_text(encoding="utf-8")
    html = html.replace(f'<link rel="stylesheet" href="{name}_style.css">', f"<style>{css}</style>")
    html = html.replace(f'<script src="{name}_script.js"></script>', f"<script>{js}</script>")
    target = Path(f"{name}.html")
    target.write_text(html, encoding="utf-8")
    shutil.rmtree(tmp.parent, ignore_errors=True)
    return target.resolve()


print("Plot geschrieben:", save_plot(flow, "changelog_flow"))   # labs/changelog_flow.html (Nebenprodukt)

## Aufgabe 6 (kurz): Human-in-the-Loop mit `@human_feedback`

**Was.** Eine Variante des Flows, in der `run_crew` zusätzlich mit `@human_feedback(message="Zusammenfassung freigeben?", emit=["approved", "rejected"], llm=llm, default_outcome="approved")` dekoriert ist, plus zwei Listener `@listen("approved")` und `@listen("rejected")`. Reihenfolge der Dekoratoren: `@listen(...)` außen, `@human_feedback(...)` innen.

**Warum.** Der Freigabeschritt ist die Stelle, an der ein Mensch in den Flow eingreift, bevor etwas nach außen geht (in Lab 4: bevor der Review-Kommentar auf GitHub landet). CrewAI zeigt die Ausgabe der Methode an und wartet auf Konsoleneingabe. Freitext wird vom `llm` auf einen der Werte in `emit` abgebildet; Enter ohne Text nimmt `default_outcome`.

**Ausführung ohne Konsole.** Der Standard-Provider ruft `input()` auf. Unter `nbconvert` (und in jeder Umgebung ohne Standardeingabe) wirft das `StdinNotImplementedError`, eine `RuntimeError`. Deshalb definieren wir einen kleinen `NotebookProvider`, der von `ConsoleProvider` erbt, `input()` versucht und bei fehlender Eingabe leeren Text zurückgibt: dann greift `default_outcome`. Im Kurs, live in Jupyter oder im Terminal, erscheint die Eingabeaufforderung, und Sie tippen Enter (Freigabe) oder einen Text wie „nein, Titel zu vage“ (das Modell klassifiziert ihn als `rejected`).

**Woran erkennen Sie Erfolg.** Im Log erscheint `OUTPUT FOR REVIEW` mit der Zusammenfassung und die Frage `Zusammenfassung freigeben?`; ohne Eingabe folgt `[approved]` mit `feedback=''`. Bei live eingegebenem Ablehnungstext läuft stattdessen `on_rejected`.

In [ ]:
from crewai.flow.human_feedback import human_feedback, HumanFeedbackResult
from crewai.flow.async_feedback import ConsoleProvider


class NotebookProvider(ConsoleProvider):
    """Wie ConsoleProvider, aber ohne stdin (nbconvert) leerer Text -> default_outcome."""
    def request_feedback(self, context, flow):
        # TODO: super().request_feedback(...) in try; bei (EOFError, RuntimeError) Hinweis drucken und "" zurückgeben
        pass


class ReviewedChangelogFlow(Flow[ChangelogState]):
    # TODO: load_diff, route, skip wie in Aufgabe 5
    # TODO: run_crew mit @listen("has_changes") außen und @human_feedback(...) innen
    # TODO: @listen("approved") on_approved(self, result: HumanFeedbackResult) und @listen("rejected") on_rejected(...)
    pass


# TODO: ReviewedChangelogFlow().kickoff(inputs={"diff_path": str(DATA / "diff_rabatt.patch")})

### Lösung

In [ ]:
# LÖSUNG
from crewai.flow.human_feedback import human_feedback, HumanFeedbackResult
from crewai.flow.async_feedback import ConsoleProvider


class NotebookProvider(ConsoleProvider):
    """Wie ConsoleProvider, aber ohne stdin (nbconvert) leerer Text -> default_outcome."""
    def request_feedback(self, context, flow):
        try:
            return super().request_feedback(context, flow)
        except (EOFError, RuntimeError) as e:   # StdinNotImplementedError ist eine RuntimeError
            print(f"[NotebookProvider] keine Eingabe möglich ({type(e).__name__}), default_outcome greift")
            return ""


class ReviewedChangelogFlow(Flow[ChangelogState]):

    @start()
    def load_diff(self):
        self.state.diff = Path(self.state.diff_path).read_text(encoding="utf-8")

    @router(load_diff)
    def route(self):
        return "empty" if not self.state.diff.strip() else "has_changes"

    @listen("has_changes")
    @human_feedback(
        message="Zusammenfassung freigeben? (Enter = ja, Text = Einwand)",
        emit=["approved", "rejected"],
        llm=llm,
        default_outcome="approved",
        provider=NotebookProvider(),
    )
    def run_crew(self):
        result = build_crew().kickoff(inputs={"diff": self.state.diff})
        self.state.summary = result.pydantic
        return json.dumps(self.state.summary.model_dump(), indent=2, ensure_ascii=False)

    @listen("approved")
    def on_approved(self, result: HumanFeedbackResult):
        print(f"[approved] feedback={result.feedback!r} -> Eintrag wird ins Changelog übernommen")
        return "approved"

    @listen("rejected")
    def on_rejected(self, result: HumanFeedbackResult):
        print(f"[rejected] feedback={result.feedback!r} -> Eintrag verworfen, Einwand an den Writer")
        return "rejected"

    @listen("empty")
    def skip(self):
        return "Kein Changelog-Eintrag: Diff ist leer."


hitl_flow = ReviewedChangelogFlow()
t0 = time.time()
outcome = hitl_flow.kickoff(inputs={"diff_path": str(DATA / "diff_rabatt.patch")})
print(f"\nFlow-Ergebnis: {outcome!r} nach {time.time() - t0:.0f} s")
print("Letztes Feedback:", hitl_flow.last_human_feedback)

## Was Sie mitnehmen

1. **Agent, Task, Crew sind drei getrennte Verträge.** Rolle und Ziel bestimmen den System-Prompt, `description` und `expected_output` den Auftrag, `context` die Datenleitung zwischen Tasks. `output_pydantic` macht aus Modelltext ein typsicheres Objekt, das der Rest des Programms verarbeiten kann.
2. **Guardrails sind deterministischer Code zwischen Modell und Weiterverarbeitung.** Sie geben eine Begründung zurück, CrewAI schickt sie dem Agenten als Feedback, und nach `guardrail_max_retries` bricht der Lauf sauber ab statt Müll weiterzureichen.
3. **Der Flow ist der Rahmen, die Crew der Motor.** Dateizugriff, Leerprüfung, Routing und Freigabe sind normaler Python-Code mit typisiertem State; das Modell läuft nur dort, wo Sprachverständnis nötig ist. Hierarchische Crews kaufen Flexibilität mit deutlich mehr Tokens und Laufzeit.

**Brücke zu Lab 4.** Dort wird aus diesem Muster die Code-Review-Pipeline: Der Flow holt PR-Metadaten und Diff über den MCP-Server aus Lab 2, eine Crew mit drei Reviewern liefert strukturierte Findings, ein Lead Reviewer führt sie zusammen, und der `@human_feedback`-Schritt entscheidet, ob der Kommentar auf den Pull Request geht.